# 2.2 Running Language Models in Python

Language models are often used through chat interfaces, but they can also be run directly from Python. Programmatic access makes it possible to incorporate language models into larger scientific workflows, where model inputs and outputs can be controlled, processed, and combined with other data sources.

In this section, we will run a small open language model and use it to answer a scientific question. We will then show how the same type of request can be made through a hosted model API as an optional alternative. In the next section, we will revisit the same question and provide the model with scientific literature as additional evidence.

## 2.2.1 Loading an Open Language Model

Open language models can be downloaded and run directly in Python rather than accessed through a remote service. In this example, we will use [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), a relatively small instruction-tuned model available through [Hugging Face](https://huggingface.co/). Its size makes it practical for a cloud notebook such as Google Colab while still demonstrating the same basic workflows used with larger language models.

First, install the libraries needed to download and run the model.

In [24]:
!pip install -q transformers accelerate ipywidgets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from transformers import pipeline

model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Device set to use mps


The `pipeline` function provides a simple interface for running pretrained models from Hugging Face. Here, we create a text-generation pipeline and load Qwen2.5-1.5B-Instruct.

The first time this cell runs, the model files will be downloaded. After loading, the `model` object can be used to send prompts to the model and return generated text.

## 2.2.2 Sending a Chat Prompt

Instruction-tuned language models are typically given a conversation made up of messages. Each message has a role, such as `system` or `user`, and some text content.

This structure is useful because it separates general isntructions for the model from the specific questions being asked. The same pattern is also used by many hosted model APIs, so learning it here makes the workflow reusable later.

In [16]:
question = (
    "How do kinase inhibitors work in cancer treatment, "
    "and what are some major limitations of this therapeutic approach?"
)

messages = [
    {
        "role": "system",
        "content": "You are a helpful scientific assistant."
    },
    {
        "role": "user",
        "content": question
    }
]

In [17]:
prompt = model.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [20]:
response = model(
    prompt,
    max_new_tokens=1024,
    do_sample=False,
)

print(response[0]["generated_text"][len(prompt):])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Kinase inhibitors are drugs that target specific enzymes called kinases, which play crucial roles in cell signaling pathways. In cancer therapy, these inhibitors can be used to block the activity of certain kinases, thereby inhibiting abnormal cellular processes that contribute to tumor growth and progression.

There are several types of kinase inhibitors available for treating various cancers:

1. Small molecule kinase inhibitors: These molecules bind to specific kinases and prevent them from interacting with their substrates or activating downstream signaling pathways.
2. Monoclonal antibodies: Some monoclonal antibodies specifically target and inhibit the function of kinases on cancer cells.
3. RNA interference (RNAi) therapies: These use small interfering RNAs to silence the expression of specific genes involved in cancer development and progression.

The mechanism by which kinase inhibitors work varies depending on the type of inhibitor and the targeted kinase. However, they gener

The model produced a clear and detailed answer without access to any sources. Its response reflects information learn during training rather than evidence supplied for this specific question. This creates an important limitation for scientific use. Even when an answer sounds plausible, individual claims may be incomplete, imprecise, or unsupported.

In the next section, we will revisit this question and provide the model with scientific literature so that its answer can be grounded in retrieved evidence.